# RF-DETR Weed Detection Fine-Tuning

Train RF-DETR on a weed dataset for whole-plant detection.

**Model:** RF-DETR Medium (54.7 mAP on COCO, ~30 FPS)

**Target Classes:**
- Dandelion
- Clover
- Crabgrass
- Dock
- Milk Thistle
- (+ other weeds from dataset)

**Requirements:**
- GPU runtime (T4 minimum, A100 recommended)
- ~2-3 hours for 50 epochs on T4 with 1000 images

In [ ]:
# Check GPU availability
!nvidia-smi

## 1. Install Dependencies

In [ ]:
# Install RF-DETR and Roboflow
!pip install -q rfdetr roboflow supervision

## 2. Download Dataset from Roboflow

You have several options for weed datasets:

1. **PWD Dataset** - ~1000 images with Dandelion, Clover, Crabgrass, Dock, Milk Thistle
2. **Augmented Startups Weeds** - 4200+ images, general garden weeds
3. **Custom Dataset** - Use your own annotated images

Search [Roboflow Universe](https://universe.roboflow.com/search?q=class:weed) for available datasets.

In [ ]:
# Login to Roboflow (will prompt for API key)
import roboflow
roboflow.login()

# Option 1: Download PWD-style dataset (update URL with your chosen dataset)
# Go to Roboflow Universe, find a weed dataset, and copy its download code

# Example using Augmented Startups Weeds dataset:
rf = roboflow.Roboflow()
project = rf.workspace("augmented-startups").project("weeds-nxe1w")
version = project.version(1)
dataset = version.download("coco")

print(f"Dataset downloaded to: {dataset.location}")

In [ ]:
# Alternative: Upload your own dataset
# If you have your own images, upload them to Roboflow first,
# annotate them, then download using the code above.
#
# Or mount Google Drive with your COCO-format dataset:
# from google.colab import drive
# drive.mount('/content/drive')
# dataset_location = '/content/drive/MyDrive/weed_dataset'

## 3. Explore Dataset

In [ ]:
import json
from pathlib import Path

# Load COCO annotations to see class names
dataset_path = Path(dataset.location)
train_annotations = dataset_path / "train" / "_annotations.coco.json"

with open(train_annotations) as f:
    coco_data = json.load(f)

# Print class names
print("Classes in dataset:")
for cat in coco_data["categories"]:
    print(f"  {cat['id']}: {cat['name']}")

print(f"\nTotal images: {len(coco_data['images'])}")
print(f"Total annotations: {len(coco_data['annotations'])}")

In [ ]:
# Visualize sample images
import supervision as sv
from PIL import Image
import matplotlib.pyplot as plt

# Load a sample image with annotations
sample_img_info = coco_data["images"][0]
sample_img_path = dataset_path / "train" / sample_img_info["file_name"]

img = Image.open(sample_img_path)
plt.figure(figsize=(10, 10))
plt.imshow(img)
plt.title(f"Sample: {sample_img_info['file_name']}")
plt.axis('off')
plt.show()

## 4. Train RF-DETR

In [ ]:
from rfdetr import RFDETRMedium

# Initialize model
model = RFDETRMedium()

# Training history callback
history = []
def log_epoch(data):
    history.append(data)
    epoch = data.get('epoch', '?')
    train_loss = data.get('train_loss', 0)
    val_loss = data.get('val_loss', 0)
    print(f"Epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

model.callbacks["on_fit_epoch_end"].append(log_epoch)

In [ ]:
# Train!
# Adjust epochs based on dataset size:
# - Small dataset (<500 images): 50-100 epochs
# - Medium dataset (500-2000 images): 30-50 epochs  
# - Large dataset (>2000 images): 15-30 epochs

model.train(
    dataset_dir=dataset.location,
    epochs=50,          # Adjust based on dataset size
    batch_size=16,      # Reduce to 8 if OOM on T4
    lr=1e-4,            # Learning rate
    device="cuda",
)

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

if history:
    epochs = [h.get('epoch', i) for i, h in enumerate(history)]
    train_losses = [h.get('train_loss', 0) for h in history]
    val_losses = [h.get('val_loss', 0) for h in history]
    
    plt.figure(figsize=(10, 5))
    plt.plot(epochs, train_losses, label='Train Loss')
    plt.plot(epochs, val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('RF-DETR Training Progress')
    plt.legend()
    plt.grid(True)
    plt.show()

## 5. Test the Trained Model

In [ ]:
# Test on a sample image
import supervision as sv
from PIL import Image

# Use a test image
test_path = dataset_path / "test"
test_images = list(test_path.glob("*.jpg")) + list(test_path.glob("*.png"))

if test_images:
    test_img = Image.open(test_images[0])
    
    # Run inference
    detections = model.predict(test_img, threshold=0.5)
    
    # Get class names from dataset
    class_names = {cat['id']: cat['name'] for cat in coco_data['categories']}
    labels = [class_names.get(cid, f"class_{cid}") for cid in detections.class_id]
    
    # Annotate
    annotator = sv.BoxAnnotator()
    label_annotator = sv.LabelAnnotator()
    
    annotated = annotator.annotate(test_img.copy(), detections)
    annotated = label_annotator.annotate(annotated, detections, labels)
    
    plt.figure(figsize=(12, 8))
    plt.imshow(annotated)
    plt.title(f"Detections: {len(detections)}")
    plt.axis('off')
    plt.show()
    
    print(f"\nDetected {len(detections)} objects:")
    for label, conf in zip(labels, detections.confidence):
        print(f"  - {label}: {conf:.2%}")
else:
    print("No test images found")

## 6. Export Trained Weights

In [ ]:
# Save the trained model
import torch

# Save weights
weights_path = "rf_detr_weed_weights.pt"
torch.save(model.model.state_dict(), weights_path)
print(f"Weights saved to: {weights_path}")

# Also save class names for inference
class_names_path = "rf_detr_weed_classes.json"
with open(class_names_path, 'w') as f:
    json.dump({
        'categories': coco_data['categories'],
        'class_names': [cat['name'] for cat in coco_data['categories']]
    }, f, indent=2)
print(f"Class names saved to: {class_names_path}")

In [ ]:
# Download weights to local machine
from google.colab import files

files.download(weights_path)
files.download(class_names_path)

print("\n" + "="*50)
print("NEXT STEPS:")
print("="*50)
print("1. Copy rf_detr_weed_weights.pt to your ai-perception/weights/ folder")
print("2. Update CLASS_NAMES in src/detection/rf_detr.py to match your dataset")
print("3. Restart the weed detection server")
print("4. Select 'RF-DETR (Weed Detection)' from the dropdown")
print("="*50)

## 7. Optional: Save to Google Drive

In [ ]:
# Mount Google Drive and save weights there for persistence
from google.colab import drive
drive.mount('/content/drive')

import shutil
drive_path = '/content/drive/MyDrive/rf_detr_weed_model/'
!mkdir -p {drive_path}

shutil.copy(weights_path, drive_path)
shutil.copy(class_names_path, drive_path)

print(f"Model saved to Google Drive: {drive_path}")